# Create RAG for projects

In [ ]:
import os
from langchain_community.document_loaders import DirectoryLoader, UnstructuredFileLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load documents
PROJECTS_DIR = "./personal_data/projects"
if not os.path.exists(PROJECTS_DIR):
    print(f"Directory {PROJECTS_DIR} does not exist.")
else:
    loader = DirectoryLoader(PROJECTS_DIR, glob="**/*.md", loader_cls=UnstructuredFileLoader)
    documents = loader.load()
    print(f"Loaded {len(documents)} documents")

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunks = splitter.split_documents(documents)
    print(f"Split into {len(chunks)} chunks")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Initialize HuggingFace Embeddings (free, local)
# Using a lightweight model suitable for small/medium datasets
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

# Set directory for persistent storage
persist_directory = "./chroma_db"

# Store documents in ChromaDB
if chunks:
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_directory
    )
    
    print(f"✅ Data successfully stored in ChromaDB at {persist_directory}!")
else:
    print("No chunks to store.")


In [ ]:
# Verify Retrieval
if os.path.isdir(persist_directory):
    vectorstore_loaded = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
    
    # Test Query
    query = "agent"
    results = vectorstore_loaded.similarity_search(query, k=2)
    
    print(f"\nTest Query: '{query}'")
    for doc in results:
        print(f"- {doc.page_content[:100]}...")
